# Lab 8b: Comparing model families and Ensemble Learning

Handwritten digits: 1797 images, each $8 \times 8$ pixels, labelled 0 to 9.


In [ ]:
# PROVIDED
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier

digits = load_digits()
X, y = digits.data, digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

cv = StratifiedKFold(5, shuffle=True, random_state=0)

print(f'images {digits.images.shape}   feature matrix {X.shape}')
print(f'train {X_train.shape[0]} rows, test {X_test.shape[0]} rows')

fig, axes = plt.subplots(2, 6, figsize=(9, 3.4))
for ax, img, label in zip(axes.ravel(), digits.images, digits.target):
    ax.imshow(img, cmap='gray_r')
    ax.set_title(str(label), fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# PROVIDED
dummy = DummyClassifier(strategy='most_frequent')
print(f'baseline CV accuracy: {cross_val_score(dummy, X_train, y_train, cv=cv).mean():.4f}')

*Your answer here.*

## 2. Seven models, same folds

Every model below is scored with the same cross-validation folds on the
training data.

In [ ]:
# PROVIDED
MODELS = [
    ('Baseline (most frequent)', DummyClassifier(strategy='most_frequent')),
    ('Linear SVM',               LinearSVC(max_iter=20000, random_state=0)),
    ('kNN (k=5)',                KNeighborsClassifier(n_neighbors=5)),
    ('RBF SVM',                  SVC(random_state=0)),
    ('Decision tree',            DecisionTreeClassifier(random_state=0)),
    ('Bagged trees',             BaggingClassifier(DecisionTreeClassifier(random_state=0),
                                                   n_estimators=100, random_state=0)),
    ('Random forest',            RandomForestClassifier(n_estimators=100, random_state=0)),
]

print(f'{"model":28s}{"train":>9s}{"CV mean":>10s}{"CV sd":>9s}')
scores = {}
for name, model in MODELS:
    s = cross_val_score(model, X_train, y_train, cv=cv)
    model.fit(X_train, y_train)
    scores[name] = s
    print(f'{name:28s}{model.score(X_train, y_train):9.4f}{s.mean():10.4f}{s.std():9.4f}')

## 3. 

In [ ]:
# PROVIDED
print(f'{"model":14s}{"raw":>10s}{"scaled":>10s}{"change":>10s}')
for name, make in [('kNN (k=5)',  lambda: KNeighborsClassifier(n_neighbors=5)),
                   ('RBF SVM',    lambda: SVC(random_state=0)),
                   ('Linear SVM', lambda: LinearSVC(max_iter=20000, random_state=0))]:
    raw = cross_val_score(make(), X_train, y_train, cv=cv).mean()
    scaled = cross_val_score(make_pipeline(StandardScaler(), make()),
                             X_train, y_train, cv=cv).mean()
    print(f'{name:14s}{raw:10.4f}{scaled:10.4f}{scaled - raw:+10.4f}')

sd = X_train.std(axis=0)
print()
print(f'pixels with standard deviation below 0.5: {(sd < 0.5).sum()} of 64')
print(f'smallest nonzero pixel standard deviation: {np.sort(sd[sd > 0])[0]:.3f}')

## 4. Why does bagging work?

In [ ]:
# PROVIDED
single = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)

print('voting 100 EXACT copies of one fitted tree:')
print(f'   one tree                      : {single.score(X_test, y_test):.4f}')
print(f'   100 identical copies, majority: {single.score(X_test, y_test):.4f}')
print()
print(f'{"ensemble":26s}{"CV mean":>10s}{"CV sd":>9s}')
for name in ['Decision tree', 'Bagged trees', 'Random forest']:
    print(f'{name:26s}{scores[name].mean():10.4f}{scores[name].std():9.4f}')

In [ ]:
# PROVIDED
forest = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_train, y_train)
tree_votes = np.array([t.predict(X_test) for t in forest.estimators_]).astype(int)

i = 0   # one test image the individual trees find hard
counts = np.bincount(tree_votes[:, i], minlength=10)

print(f'test image {i}, true label {y_test[i]}')
print(f'   first five trees vote : {tree_votes[:5, i].tolist()}')
print(f'   all 100 trees         : {dict((d, int(c)) for d, c in enumerate(counts) if c)}')
print(f'   the forest answers    : {forest.predict(X_test[i:i + 1])[0]}')
print()
per_tree = np.mean([(tree_votes[j] == y_test).mean() for j in range(100)])
print(f'average accuracy of ONE tree in the forest : {per_tree:.4f}')
print(f'accuracy of the whole forest               : {forest.score(X_test, y_test):.4f}')

fig, ax = plt.subplots(figsize=(1.8, 1.8))
ax.imshow(X_test[i].reshape(8, 8), cmap='gray_r')
ax.set_title(f'true: {y_test[i]}', fontsize=10)
ax.set_xticks([]); ax.set_yticks([])
plt.show()

**Q5.** Voting 100 identical copies of one tree changes nothing. Bagging 100
trees gains about nine points. What is different about the 100 trees in the
second case?

**Q6.** The average single tree in the forest is far worse than the standalone
decision tree from section 2, yet the forest beats everything. How can
combining weaker models give a better one?

## 5. Deploy model

Pick the model you would deploy, based on the cross-validation table in section
2. Fit it on the full training set and score it on the test set.

In [ ]:
final = SVC(random_state=0).fit(X_train, y_train)
predictions = final.predict(X_test)

print(f'test accuracy: {final.score(X_test, y_test):.4f}')
print(f'misclassified: {(predictions != y_test).sum()} of {len(y_test)} images')

In [ ]:
# PROVIDED  (run this after the cell above; it uses `final` and `predictions`)
fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay.from_predictions(y_test, predictions, ax=ax, colorbar=False,
                                        cmap='Blues', values_format='d')
ax.set_title('Confusion matrix on the test set', fontsize=11)
plt.tight_layout(); plt.show()

wrong = np.where(predictions != y_test)[0]
if len(wrong):
    fig, axes = plt.subplots(1, len(wrong), figsize=(1.7 * len(wrong), 2.1))
    for ax, idx in zip(np.atleast_1d(axes), wrong):
        ax.imshow(X_test[idx].reshape(8, 8), cmap='gray_r')
        ax.set_title(f'true {y_test[idx]}\npredicted {predictions[idx]}', fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()
else:
    print('no misclassified test images')

**Q7.** Look at the images the model got wrong. Pick one and say why you think
it was hard.